# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `release_events` Silver Table

This step creates the Delta table used to store cleaned and structured **ReleaseEvent** records in the Silver layer.

The table contains three groups of fields:

- **Common event fields** — General GitHub event information including event ID, timestamp, actor, and repository.
- **Bronze lineage fields** — Metadata carried forward from the Bronze layer to preserve data lineage.
- **ReleaseEvent-specific fields** — Information describing repository releases, including the release action, release ID, tag, publication timestamp, draft status, and prerelease status.

These fields allow the Silver layer to distinguish normal releases, draft releases, and prereleases while also tracking when releases were published.

This structured data can later support analysis of repository release activity and software delivery patterns.

The table is stored using **Delta Lake** (`USING DELTA`).

`CREATE TABLE IF NOT EXISTS` prevents an existing table from being recreated when the notebook is rerun.

> **Note:** This step defines only the destination structure. ReleaseEvent extraction, cleaning, validation, and loading from the Bronze layer are performed in subsequent steps.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.release_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Event-specific fields
    release_action        STRING,
    release_id            BIGINT,
    release_tag           STRING,
    release_published_at  TIMESTAMP,
    release_draft         BOOLEAN,
    release_prerelease    BOOLEAN,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP
)
USING DELTA;